# Mat & Comp – RN 4: PyTorch con MNIST
## Una red que lee 70 000 dígitos

**Curso:** Redes neuronales (Sección 4) · **Duración sugerida:** 2 sesiones de 50 minutos · **Requisito:** RN 3

### Objetivos de aprendizaje
Al terminar esta lección podrás:
1. Crear y operar **tensores** de PyTorch y relacionarlos con los arreglos de numpy.
2. Usar **autograd** para que PyTorch calcule las pendientes solo (la retropropagación de RN 3, automática).
3. Entrenar con descenso de gradiente usando `nn.Linear`, una **función de pérdida** y un **optimizador**.
4. Construir y entrenar una red de unos **100 000 parámetros** que reconoce los dígitos de **MNIST** con cerca de 97 % de exactitud.
5. Evaluar la red con la **matriz de confusión** y descubrir una debilidad que motiva las redes convolucionales.

## 🔍 ¿Cuántas cuentas?

En RN 3 escribimos a mano la retropropagación de una red de 2410 parámetros para dígitos de 8 × 8 píxeles. Ahora queremos dígitos de **28 × 28** píxeles, con una red de

$$784 \times 128 + 128 + 128 \times 10 + 10 = 101\,770 \text{ parámetros.}$$

✏️ *Si cada pendiente se midiera «a lo bruto», moviendo un parámetro y recalculando el error en 60 000 imágenes (como en RN 2), ¿cuántas veces habría que recorrer las 60 000 imágenes en **cada** paso del entrenamiento? ¿Y si además cambiamos la red, tendríamos que volver a deducir todas las fórmulas de RN 3?*

**Mi respuesta:**

Hoy usaremos **PyTorch**, la librería con la que se construyen muchas de las redes neuronales modernas. Tú describes la red; PyTorch hace la retropropagación **automáticamente**.

## 1. Tensores: los arreglos de PyTorch

Un **tensor** es casi lo mismo que un arreglo de numpy (Python 11): una tabla de números de cualquier dimensión, con operaciones elemento a elemento y producto de matrices `@`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([10.0, 20.0, 30.0])
print("a + b =", a + b)
print("a * b =", a * b)
print("a @ b =", a @ b)       # producto punto

M = torch.tensor([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]])
print("M tiene forma", M.shape)
print("Suma de cada columna:", M.sum(dim=0))   # dim hace el papel de axis

Pasar de numpy a PyTorch y de vuelta es inmediato:

In [ ]:
arr = np.arange(6).reshape(2, 3)
t = torch.tensor(arr, dtype=torch.float32)   # de numpy a tensor
print(t)
print(t.numpy())                             # de tensor a numpy

| numpy | PyTorch |
|---|---|
| `np.array([...])` | `torch.tensor([...])` |
| `A.shape` | `A.shape` |
| `A.sum(axis=0)` | `A.sum(dim=0)` |
| `A @ B` | `A @ B` |
| `A.reshape(...)` | `A.reshape(...)` |
| `np.random.randn(3, 2)` | `torch.randn(3, 2)` |

## 2. Autograd: pendientes automáticas

En RN 3 calculamos a mano, con la regla de la cadena, la pendiente de $y = (3x + 1)^2$ en $x = 1$: dio $2 \cdot 4 \cdot 3 = 24$.

En PyTorch basta con marcar `x` con `requires_grad=True`. PyTorch **anota** cada operación que haces con `x`, y al llamar `.backward()` recorre las anotaciones hacia atrás, igual que nuestra retropropagación:

In [ ]:
x = torch.tensor(1.0, requires_grad=True)
y = (3 * x + 1) ** 2
y.backward()          # retropropagación automática
print("y =", y.item())
print("dy/dx =", x.grad.item())

Funciona con cualquier cadena de operaciones. Repitamos la **neurona** de RN 3: entrada $x = 2$, respuesta correcta $y = 1$, peso $w = -0.5$ y sesgo $b = 0.3$ (la de la sección 3 de RN 3).

In [ ]:
w = torch.tensor(-0.5, requires_grad=True)
b = torch.tensor(0.3, requires_grad=True)
x, y = 2.0, 1.0

a = torch.sigmoid(w * x + b)     # la neurona
E = (a - y) ** 2                 # el error
E.backward()

print(f"a = {a.item():.4f}   E = {E.item():.4f}")
print(f"dE/dw = {w.grad.item():.4f}   dE/db = {b.grad.item():.4f}")

# Comparemos con la regla de la cadena de RN 3:
a_ = a.item()
print(f"A mano: dE/dw = {2 * (a_ - y) * a_ * (1 - a_) * x:.4f}   dE/db = {2 * (a_ - y) * a_ * (1 - a_):.4f}")

¡Las mismas pendientes, sin escribir ninguna derivada! 🎉

## 3. Descenso de gradiente con autograd

Volvamos a los datos de ML 2 y ML 3: predecir la **nota de matemáticas** a partir de las **horas de estudio** con una recta $\hat{y} = w x + b$. Primero, como en ML 3, con un ciclo escrito por nosotros; solo que ahora las pendientes las da `backward()`.

In [ ]:
import pandas as pd

URL = "https://raw.githubusercontent.com/emersonjleon/compumath/main/lecciones/seccion3_ml/datos/estudiantes.csv"
datos = pd.read_csv(URL)
x = torch.tensor(datos["horas_estudio"].values, dtype=torch.float32)
y = torch.tensor(datos["nota_matematicas"].values, dtype=torch.float32)
print(len(x), "estudiantes")

In [ ]:
w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
alfa = 0.01

for paso in range(3001):
    pred = w * x + b
    E = ((pred - y) ** 2).mean()   # error cuadrático medio
    E.backward()                   # calcula w.grad y b.grad
    with torch.no_grad():          # actualizar sin anotar estas cuentas
        w -= alfa * w.grad
        b -= alfa * b.grad
    w.grad.zero_()                 # borrar las pendientes para el siguiente paso
    b.grad.zero_()
    if paso % 500 == 0:
        print(f"paso {paso:4}: E = {E.item():.4f}   w = {w.item():.3f}   b = {b.item():.3f}")

pendiente, corte = np.polyfit(datos["horas_estudio"], datos["nota_matematicas"], 1)
print(f"\nMínimos cuadrados (ML 2): w = {pendiente:.3f}   b = {corte:.3f}")

Llegamos a la misma recta que los mínimos cuadrados de ML 2. Fíjate en tres detalles nuevos:

- `with torch.no_grad():` le dice a PyTorch que la actualización de los pesos **no** es parte de la cadena que debe anotar.
- `w.grad.zero_()` borra la pendiente. **PyTorch suma** las pendientes nuevas a las viejas; si no las borramos, se acumulan (lo veremos en *Errores comunes*).
- El guion bajo al final de `zero_` significa «modifica el tensor en su lugar».

### La versión «de librería»

PyTorch trae piezas listas para cada parte del ciclo:

| Pieza | Qué hace | Nosotros antes |
|---|---|---|
| `nn.Linear(1, 1)` | una capa: guarda `w` y `b` y calcula `x @ W + b` | `w * x + b` |
| `nn.MSELoss()` | la función de pérdida (error cuadrático medio) | `((pred - y) ** 2).mean()` |
| `optim.SGD(..., lr=alfa)` | el optimizador: aplica el descenso de gradiente | `w -= alfa * w.grad` |
| `optimizador.zero_grad()` | borra las pendientes | `w.grad.zero_()` |

In [ ]:
from torch import nn, optim

torch.manual_seed(0)
modelo = nn.Linear(1, 1)                       # 1 entrada, 1 salida
criterio = nn.MSELoss()
optimizador = optim.SGD(modelo.parameters(), lr=0.01)

X = x.reshape(-1, 1)    # nn.Linear quiere una fila por ejemplo: forma (n, 1)
Y = y.reshape(-1, 1)

for paso in range(3001):
    pred = modelo(X)
    perdida = criterio(pred, Y)
    optimizador.zero_grad()
    perdida.backward()
    optimizador.step()

print(f"w = {modelo.weight.item():.3f}   b = {modelo.bias.item():.3f}")

Estas cuatro líneas (`modelo(X)`, `criterio`, `zero_grad`, `backward`, `step`) son el **corazón de todo entrenamiento en PyTorch**, desde esta recta hasta las redes más grandes. Apréndelas de memoria. 🧠

## 4. MNIST: 70 000 dígitos escritos a mano

**MNIST** es el conjunto de datos más famoso del machine learning: 60 000 imágenes de entrenamiento y 10 000 de prueba, cada una de 28 × 28 píxeles en escala de grises. La librería `torchvision` (incluida en Colab) lo descarga en unos segundos.

In [ ]:
from torchvision import datasets, transforms

entreno = datasets.MNIST("datos", train=True, download=True, transform=transforms.ToTensor())
prueba = datasets.MNIST("datos", train=False, download=True, transform=transforms.ToTensor())
print(len(entreno), "imágenes de entrenamiento y", len(prueba), "de prueba")

`transforms.ToTensor()` convierte cada imagen en un tensor con valores entre 0 y 1 (así ya queda **normalizada**, ¡recuerda el error 4 de RN 3!).

In [ ]:
imagen, etiqueta = entreno[0]
print("Forma de la imagen:", imagen.shape)    # (canales, alto, ancho)
print("Etiqueta:", etiqueta)
print("Valores entre", imagen.min().item(), "y", imagen.max().item())

fig, ejes = plt.subplots(2, 8, figsize=(12, 3.4))
for i, eje in enumerate(ejes.flat):
    img, et = entreno[i]
    eje.imshow(img[0], cmap="gray_r")
    eje.set_title(et)
    eje.axis("off")
plt.suptitle("Las primeras 16 imágenes de MNIST")
plt.show()

✏️ *La forma es `(1, 28, 28)`: 1 canal (gris), 28 filas y 28 columnas. ¿Cuántos números describen cada imagen? ¿Cuántos tendría una imagen a color, con 3 canales?*

### Lotes (mini-batches)

Con 60 000 imágenes no conviene calcular el error con **todas** antes de cada paso: sería lento. En cambio, tomamos **lotes** de 64 imágenes al azar y damos un paso con cada lote. A esto se le llama **descenso de gradiente estocástico**. Una **época** es una pasada completa por las 60 000 imágenes (unos 938 lotes).

El `DataLoader` reparte las imágenes en lotes:

In [ ]:
from torch.utils.data import DataLoader

torch.manual_seed(0)
cargador_entreno = DataLoader(entreno, batch_size=64, shuffle=True)
cargador_prueba = DataLoader(prueba, batch_size=1000)

X_lote, y_lote = next(iter(cargador_entreno))
print("Un lote de imágenes:", X_lote.shape)
print("Sus etiquetas:", y_lote.shape, y_lote[:10])
print("Lotes por época:", len(cargador_entreno))

## 5. La red: 784 → 128 → 10

Nuestra red tendrá:

1. `nn.Flatten()`: «aplana» cada imagen de 28 × 28 en una fila de 784 números.
2. `nn.Linear(784, 128)` y luego `nn.ReLU()`: una capa oculta de 128 neuronas con la activación **ReLU**, $\max(0, z)$, que evita el desvanecimiento del gradiente (RN 3).
3. `nn.Linear(128, 10)`: una salida por cada dígito, del 0 al 9.

`nn.Sequential` encadena las capas en orden:

In [ ]:
torch.manual_seed(0)
red = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 128),
    nn.ReLU(),
    nn.Linear(128, 10),
)
print(red)

total = sum(p.numel() for p in red.parameters())
print(f"\nParámetros: {total:,}")
for nombre, p in red.named_parameters():
    print(f"  {nombre:10} forma {tuple(p.shape)}")

Exactamente los 101 770 parámetros que calculamos al inicio. (PyTorch guarda cada matriz de pesos «al revés»: `(128, 784)` en lugar de `(784, 128)`, pero la idea es la misma de RN 2.)

### Puntajes, probabilidades y la pérdida de entropía cruzada

La red da **10 puntajes** por imagen (se llaman *logits*). El más alto es su respuesta. La función **softmax** los convierte en probabilidades que suman 1:

$$p_k = \frac{e^{s_k}}{e^{s_0} + e^{s_1} + \cdots + e^{s_9}}$$

Para clasificar no usaremos el error cuadrático, sino la **entropía cruzada**: si la probabilidad que la red da al dígito correcto es $p$, la pérdida es $-\ln p$. Si $p$ es casi 1, la pérdida es casi 0; si $p$ es pequeña, la pérdida es grande.

In [ ]:
p = np.linspace(0.01, 1, 200)
plt.figure(figsize=(5, 3))
plt.plot(p, -np.log(p))
plt.xlabel("probabilidad dada al dígito correcto")
plt.ylabel("pérdida  −ln p")
plt.title("Entropía cruzada")
plt.grid(alpha=0.3)
plt.show()

with torch.no_grad():
    puntajes = red(X_lote[:1])          # la red aún no ha aprendido
probs = torch.softmax(puntajes, dim=1)
print("Puntajes:", puntajes.numpy().round(2))
print("Probabilidades:", probs.numpy().round(3))
print("Suman:", probs.sum().item())

Antes de entrenar, la red reparte las probabilidades casi por igual (cerca de 0.1 cada una): está adivinando. En PyTorch, `nn.CrossEntropyLoss()` aplica la softmax **y** el $-\ln p$ a la vez, así que la red debe entregar los puntajes **sin** softmax.

## 6. ¡A entrenar!

El ciclo es el mismo de la sección 3, ahora recorriendo los lotes:

In [ ]:
criterio = nn.CrossEntropyLoss()
optimizador = optim.SGD(red.parameters(), lr=0.1)

def entrenar_una_epoca(red, cargador):
    red.train()
    suma = 0
    for X, y in cargador:
        perdida = criterio(red(X), y)
        optimizador.zero_grad()
        perdida.backward()
        optimizador.step()
        suma += perdida.item() * len(X)
    return suma / len(cargador.dataset)

def exactitud(red, cargador):
    red.eval()
    correctos = 0
    with torch.no_grad():
        for X, y in cargador:
            correctos += (red(X).argmax(dim=1) == y).sum().item()
    return correctos / len(cargador.dataset)

print(f"Antes de entrenar: exactitud en prueba = {exactitud(red, cargador_prueba):.1%}")

In [ ]:
import time

perdidas, exactitudes = [], []
inicio = time.time()
for epoca in range(1, 6):
    perd = entrenar_una_epoca(red, cargador_entreno)
    exa = exactitud(red, cargador_prueba)
    perdidas.append(perd)
    exactitudes.append(exa)
    print(f"Época {epoca}: pérdida = {perd:.4f}   exactitud en prueba = {exa:.1%}   ({time.time() - inicio:.0f} s)")

In [ ]:
fig, (e1, e2) = plt.subplots(1, 2, figsize=(10, 3.2))
e1.plot(range(1, 6), perdidas, "o-")
e1.set_xlabel("época"); e1.set_ylabel("pérdida promedio"); e1.set_title("Pérdida en entrenamiento")
e2.plot(range(1, 6), [100 * e for e in exactitudes], "o-", color="tab:green")
e2.set_xlabel("época"); e2.set_ylabel("% de aciertos"); e2.set_title("Exactitud en prueba")
for e in (e1, e2):
    e.grid(alpha=0.3)
    e.set_xticks(range(1, 6))
plt.tight_layout()
plt.show()

En menos de un minuto, la red acierta cerca del **97 %** de 10 000 dígitos que **nunca vio** durante el entrenamiento. (Tus números pueden variar un poco: el azar de los lotes cambia de una computadora a otra.)

✏️ *Compara con RN 3: allí el entrenamiento era un ciclo que escribimos nosotros con la retropropagación a mano. ¿Qué parte de ese trabajo hizo hoy PyTorch?*

## 7. ¿En qué se equivoca?

Veamos las probabilidades que asigna la red a algunas imágenes de prueba:

In [ ]:
X_pru, y_pru = next(iter(cargador_prueba))     # las primeras 1000 imágenes de prueba
red.eval()
with torch.no_grad():
    probs = torch.softmax(red(X_pru), dim=1)
pred = probs.argmax(dim=1)

fig, ejes = plt.subplots(2, 4, figsize=(12, 4.5), gridspec_kw={"height_ratios": [1.3, 1]})
for j, i in enumerate([0, 1, 2, 3]):
    ejes[0, j].imshow(X_pru[i, 0], cmap="gray_r")
    ejes[0, j].set_title(f"real {y_pru[i].item()}, dijo {pred[i].item()}")
    ejes[0, j].axis("off")
    ejes[1, j].bar(range(10), probs[i].numpy())
    ejes[1, j].set_xticks(range(10))
    ejes[1, j].set_ylim(0, 1)
plt.tight_layout()
plt.show()

Ahora, algunas de las imágenes en las que se **equivocó**:

In [ ]:
errores = torch.nonzero(pred != y_pru).flatten()
print(f"Errores en estas 1000 imágenes: {len(errores)}")

fig, ejes = plt.subplots(1, 8, figsize=(14, 2.5))
for eje, i in zip(ejes, errores[:8]):
    eje.imshow(X_pru[i, 0], cmap="gray_r")
    eje.set_title(f"real {y_pru[i].item()}, dijo {pred[i].item()}")
    eje.axis("off")
plt.show()

✏️ *¿Te habrías equivocado tú con alguna? ¿Hay alguna que parezca más bien un error de quien escribió el dígito?*

### La matriz de confusión

La **matriz de confusión** (ML 7) cuenta, para cada dígito real (fila), qué respondió la red (columna). En la diagonal están los aciertos.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

reales, predichas = [], []
with torch.no_grad():
    for X, y in cargador_prueba:
        reales.append(y)
        predichas.append(red(X).argmax(dim=1))
reales = torch.cat(reales).numpy()
predichas = torch.cat(predichas).numpy()

matriz = confusion_matrix(reales, predichas)
fig, eje = plt.subplots(figsize=(6.5, 6.5))
ConfusionMatrixDisplay(matriz).plot(ax=eje, cmap="Blues", colorbar=False)
eje.set_xlabel("respuesta de la red"); eje.set_ylabel("dígito real")
plt.show()

✏️ *Busca fuera de la diagonal el número más grande. ¿Qué par de dígitos confunde más la red? ¿Tiene sentido por su forma?*

## 8. ⚠️ Una debilidad: mover la imagen

Para la red, una imagen es solo una fila de 784 números: el píxel 300 es «el píxel 300», sin saber quiénes son sus vecinos. ¿Qué pasa si **corremos** un dígito unos píxeles a la derecha? Para nosotros sigue siendo el mismo dígito…

In [ ]:
i = 0
fig, ejes = plt.subplots(1, 6, figsize=(14, 2.8))
for eje, d in zip(ejes, [0, 1, 2, 3, 4, 5]):
    movida = torch.roll(X_pru[i:i+1], shifts=d, dims=3)   # corre d píxeles a la derecha
    with torch.no_grad():
        p = torch.softmax(red(movida), dim=1)[0]
    eje.imshow(movida[0, 0], cmap="gray_r")
    eje.set_title(f"+{d} px: dijo {p.argmax().item()} ({p.max().item():.0%})")
    eje.axis("off")
plt.show()

Con unos pocos píxeles de desplazamiento la red empieza a dudar o se equivoca, aunque el dígito es el mismo. La red **memorizó posiciones**, no formas. En la próxima lección veremos las **redes convolucionales**, que miran la imagen por pedacitos y reconocen una forma esté donde esté.

## 9. Errores comunes

**Error 1: olvidar `zero_grad`.** PyTorch **suma** las pendientes nuevas a las que ya había. Si llamamos `backward()` dos veces sin borrar:

In [ ]:
x = torch.tensor(1.0, requires_grad=True)
for vez in range(3):
    y = (3 * x + 1) ** 2
    y.backward()
    print(f"Vez {vez + 1}: x.grad = {x.grad.item()}")

La pendiente correcta es 24, pero se acumula: 24, 48, 72… En un entrenamiento sin `optimizador.zero_grad()`, los pasos se vuelven cada vez más grandes y el aprendizaje se descontrola.

**Error 2: formas que no encajan.** Si quitamos el `nn.Flatten()`, la capa `nn.Linear(784, 128)` recibe imágenes de 28 × 28 en lugar de filas de 784:

In [ ]:
sin_aplanar = nn.Sequential(nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 10))
sin_aplanar(X_lote)

Lee el mensaje: `mat1 and mat2 shapes cannot be multiplied` es el mismo problema de formas de las matrices de RN 2 y RN 3. Imprimir `.shape` en cada paso es el mejor remedio.

**Error 3: poner softmax al final de la red.** `nn.CrossEntropyLoss` ya aplica la softmax. Si la red también la aplica, se aplica dos veces y la red aprende mucho más lento.

**Error 4: usar un tensor con `requires_grad` como número de Python.** Para imprimir o guardar un valor usa `.item()` (un número) o `.detach().numpy()` (un arreglo).

**Error 5: actualizar pesos a mano sin `torch.no_grad()`.** PyTorch intentaría anotar la actualización como parte de la cadena y da error.

## 🤔 ¿Qué pasa si…?

Usa la celda de abajo. Antes de ejecutar, **predice** el resultado. (Para volver a entrenar desde cero, vuelve a ejecutar la celda que crea `red` y la que crea el `optimizador`.)

1. …calculas con autograd la pendiente de `torch.relu(x)` en `x = 3.0` y en `x = -3.0`?
2. …usas `lr=10` en el optimizador de la recta (sección 3)?
3. …cambias `batch_size=64` por `batch_size=1000`? ¿Cuántos lotes hay por época? ¿Sube o baja la exactitud después de una época?
4. …cambias `nn.ReLU()` por `nn.Sigmoid()` y entrenas una época?
5. …le das a la red una imagen completamente en blanco, `torch.zeros(1, 1, 28, 28)`? ¿Qué dígito dice y con qué probabilidad?

In [ ]:
# Prueba aquí tus ideas

## ✍️ Ejercicios

**Ejercicio 1 (fácil). Autograd.** En RN 3 calculaste a mano la pendiente de $y = (2x + 3)^3$ en $x = 1$. Calcúlala ahora con autograd y comprueba que coincide.

**Ejercicio 2 (fácil). Contar parámetros.** ¿Cuántos parámetros tiene una red 784 → 64 → 10? Calcúlalo a mano y compruébalo construyéndola con `nn.Sequential` y contando con `sum(p.numel() for p in red.parameters())`.

**Ejercicio 3 (medio). ¿Cuántas neuronas ocultas?** Entrena redes con 16, 64 y 256 neuronas ocultas durante **una** época cada una (`lr=0.1`, lotes de 64). Anota la exactitud en prueba y el tiempo. ¿Vale la pena tener más neuronas?

**Ejercicio 4 (medio). Una capa más.** Construye una red 784 → 128 → 64 → 10 (con ReLU después de cada capa oculta). ¿Cuántos parámetros tiene? Entrénala 5 épocas y compara su exactitud con la red de la sección 6.

**Ejercicio 5 (avanzado). Los pares más confundidos.** Usando la matriz de confusión de la sección 7, escribe código que encuentre los **3 pares** (real, respuesta) con más errores. Pista: copia la matriz, pon la diagonal en 0 con `np.fill_diagonal` y busca los valores más grandes con `np.argsort` sobre `matriz.flatten()`. Muestra 5 imágenes de cada par.

## 🏆 Reto opcional: ¿cuánto aguanta el desplazamiento?

1. Calcula la exactitud de la red sobre **todo** el conjunto de prueba con las imágenes corridas 0, 1, 2, 3, 4 y 5 píxeles a la derecha (usa `torch.roll(X, shifts=d, dims=3)` dentro de un ciclo sobre `cargador_prueba`), y grafica la exactitud contra el desplazamiento.
2. **Aumento de datos:** entrena una red nueva en la que, a cada lote, corres las imágenes un número **al azar** de píxeles entre −3 y 3. ¿Mejora la exactitud con imágenes corridas? ¿Qué pasa con la exactitud sin correr?

## 🎬 Cierre

Hoy aprendiste que:
- Un **tensor** de PyTorch es como un arreglo de numpy que además puede **anotar** sus operaciones.
- **Autograd** (`backward()`) hace la retropropagación de RN 3 automáticamente para cualquier cadena de operaciones.
- Todo entrenamiento en PyTorch sigue el mismo ciclo: **predecir, medir la pérdida, `zero_grad`, `backward`, `step`**.
- Para clasificar se usa la **entropía cruzada**, y se entrena por **lotes** recorriendo los datos varias **épocas**.
- Una red de 101 770 parámetros aprende a leer los dígitos de MNIST con cerca de 97 % de aciertos en menos de un minuto.
- Pero esa red no entiende las formas: si corremos la imagen unos píxeles, se confunde.

**Para pensar:** cuando tú miras un dígito, buscas trazos: una curva arriba, una línea vertical, un círculo cerrado… sin importar en qué parte de la hoja estén. ¿Podría una red buscar esos mismos trazos en **cada** pedacito de la imagen? Esa es la idea de las **redes convolucionales**, que construiremos con PyTorch en la próxima lección. 🔎